#### READ THE ORDER REVIEWS DATA FROM VOLUME

In [0]:
import uuid, traceback
from datetime import datetime
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
BASE_DIR = '/Volumes/orders_sales/landing/operational/OrdersReviews/'
CHECKPOINT_DIR = '/Volumes/orders_sales/landing/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/order_reviews'
CATALOG = 'ORDERS_SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'ORDERS_REVIEWS'
TARGET = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"
AUDIT_TABLE = f"{CATALOG}.AUDIT.PIPELINE_AUDIT_LOG"
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
order_reviews_schema = """review_id STRING, 
                 order_id STRING, 
                 review_score INT, 
                 review_comment_title STRING, 
                 review_comment_message STRING"""

#### LOAD THE DATA INTO BRONZE LAYER
- `ORDERS_SALES.BRONZE.ORDERS_REVIEWS`

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import builtins

def write_audit(run_id, status, start_time, rows=None, error=None):
    end = datetime.now()
    row = (run_id, "ORDER_REVIEWS_LOAD", "BRONZE", BASE_DIR, TARGET, status, rows,
           start_time, end, (end - start_time).total_seconds(),
           error[:4000] if error else None,
           spark.sql("SELECT current_user()").first()[0])
    (spark.createDataFrame([row], spark.table(AUDIT_TABLE).schema)
          .write.mode("append").format("delta").saveAsTable(AUDIT_TABLE))

def current_version(table):
    try:
        return DeltaTable.forName(spark, table).history(1).select("version").first()[0]
    except Exception:          # table doesn't exist yet (first run)
        return -1

run_id, start_time = str(uuid.uuid4()), datetime.now()
status, rows_written, err = "SUCCESS", None, None

try:
    version_before = current_version(TARGET)
    order_reviews_df = (
                spark.readStream.format('cloudFiles')
                    .option('cloudFiles.format', 'csv')
                    .option('cloudFiles.schemaEvolutionMode', 'rescue')
                    .option('header', 'true')
                    .schema(order_reviews_schema)
                    .load(f'{BASE_DIR}')
            )
    order_reviews_trans_df = order_reviews_df.select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "_metadata.file_name",
        "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

    qry = (
        order_reviews_trans_df.writeStream.format('delta')
        .outputMode('append')
        .option('checkpointLocation', CHECKPOINT_LOCATION)
        .trigger(availableNow=True)
        .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
    )
    qry.awaitTermination()

    # rows committed by this run only
    rows_written = int(
        DeltaTable.forName(spark, TARGET).history()
            .filter((F.col("version") > version_before) &
                    (F.col("operation") == "STREAMING UPDATE"))
            .select(F.col("operationMetrics")["numOutputRows"].cast("long").alias("n"))
            .agg(F.coalesce(F.sum("n"), F.lit(0))).first()[0]
    )

except Exception:
    status, err = "FAILED", traceback.format_exc()
    raise
finally:
    write_audit(run_id, status, start_time, rows_written, err)